In [126]:
import torch
import torch.nn as nn
import math
import tiktoken
from pathlib import Path

In [127]:
VOCAB_SIZE = 50257
BLOCK_SIZE = 256
N_EMBD = 128

N_HEAD = 4
HEAD_SIZE = 32

MLP_HIDEN_DIM = 4
N_TRANSFORMER_BLOCK = 4
VAL_DATA_RATIO = 0.1

In [128]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# Embeddings

In [129]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        self.block_size = block_size
        self.tok_embedding = nn.Embedding(vocab_size, n_embd, device=device)
        self.pos_embedding = nn.Embedding(block_size, n_embd, device=device)


    def forward(self, x):
        size = x.size(dim=-1)
        if (size > self.block_size):
            raise ValueError(f"Input size ({size}) exceded max block size ({self.block_size})")

        idx = torch.arange(size).to(device)

        pos_emb = self.pos_embedding(idx)
        tok_emb = self.tok_embedding(x)

        emb = tok_emb + pos_emb

        return emb


# Self Attention

In [130]:
class SelfAttention(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.head_size = head_size

        self.Q = nn.Linear(n_embd, head_size, device=device)
        self.K = nn.Linear(n_embd, head_size, device=device)
        self.V = nn.Linear(n_embd, head_size, device=device)

    def forward(self, x):
        q = self.Q(x)
        k = self.K(x)
        v = self.V(x)

        attention_score = (q @ k.transpose(-1,-2)) / math.sqrt(self.head_size)

        causal_mask = torch.tril(
            torch.ones(
                (attention_score.size(dim=-1), 
                 attention_score.size(dim=-2)), 
                 dtype=torch.bool,
                 device=device
            )
        )

        masked_attention_score = attention_score.masked_fill(~causal_mask, -torch.inf)
        score = torch.softmax(masked_attention_score, dim=-1)

        res = score @ v
        return res

In [131]:
emb = Embedding(VOCAB_SIZE, BLOCK_SIZE, N_EMBD)
x = emb(torch.randint(0, 10, (4, 200), device=device))
print(x.shape)

att = SelfAttention(N_EMBD, HEAD_SIZE)
att(x).shape

torch.Size([4, 200, 128])


torch.Size([4, 200, 32])

# Multi Head Attention

In [132]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        if (n_embd % n_head != 0):
            raise ValueError(f"Number of embeding ({n_embd}) not divisible by number of head ({n_head})")

        self.heads = nn.ModuleList(
            [SelfAttention(n_embd, int(n_embd / n_head)) for _ in range(n_head)]
        )
        self.linear = nn.Linear(n_embd, n_embd, device=device)

    def forward(self, x):
        attentions = []

        for head in self.heads:
            attentions.append(head(x))

        attention_cat = torch.cat(attentions, dim=-1)
        res = self.linear(attention_cat)

        return res


In [133]:
emb = Embedding(VOCAB_SIZE, BLOCK_SIZE, N_EMBD)
x = emb(torch.randint(0, 10, (4, 200), device=device))
print(x.shape)
#print(N_EMBD, HEAD_SIZE)
att = MultiHeadAttention(N_EMBD, N_HEAD)
att(x).shape

torch.Size([4, 200, 128])


torch.Size([4, 200, 128])

# Feed Forward MLP

In [134]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_embd, n_embd * MLP_HIDEN_DIM, device=device),
            nn.GELU(),
            nn.Linear(n_embd * MLP_HIDEN_DIM, n_embd, device=device)
        )

    def forward(self, x):
        res = self.layers(x)

        return res

# Transformer

In [135]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        self.m_heads_attention = MultiHeadAttention(n_embd, n_head)
        self.mlp = FeedForward(n_embd)
        self.heads_norm = nn.LayerNorm(n_embd, device=device)
        self.mlp_norm = nn.LayerNorm(n_embd, device=device)


    def forward(self, x):
        norm1 = self.heads_norm(x)
        attention = self.m_heads_attention(norm1)
        residual1 = x + attention

        norm2 = self.mlp_norm(residual1)
        mlp = self.mlp(norm2)
        residual2 = residual1 + mlp

        return residual2


# LLM

In [136]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
class LLM(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=N_EMBD, n_head=N_HEAD, n_layer=N_TRANSFORMER_BLOCK):
        super().__init__()

        self.block_size = block_size
        self.emb = Embedding(vocab_size, block_size, n_embd)

        transformer_blocks = [TransformerBlock(n_embd, n_head) for _ in range(n_layer)]
        self.transformer_blocks = nn.Sequential(*transformer_blocks)

        self.norm = nn.LayerNorm(n_embd, device=device)
        self.token_layer = nn.Linear(n_embd, vocab_size, device=device)

    def generate(self, idx, max_new_token, temperature=1.0):
        if (temperature <= 0):
            raise ValueError(f"Temperature ({temperature}) must be greater than 0")

        self.eval()

        tokens = idx.clone()
        with torch.no_grad():
            for _ in range(max_new_token):
                logits, _ = self(tokens[:, -self.block_size:])
                probs = torch.softmax(logits[:, -1] / temperature, dim=-1)

                new_token = torch.multinomial(probs, num_samples=1)
                tokens = torch.cat([tokens, new_token], dim=-1)

        return tokens

    def forward(self, x, targets=None):
        emb = self.emb(x)

        transformer = self.transformer_blocks(emb)
        norm = self.norm(transformer)

        logits = self.token_layer(norm)

        if (targets is None):
            return logits, None

        loss = loss_fn(logits.reshape((-1, logits.size(dim=-1))), targets.reshape(-1))
        return logits, loss

In [138]:
x = torch.randint(0, 10, (4, 128), device=device)
gpt2 = LLM(VOCAB_SIZE, BLOCK_SIZE, N_EMBD, N_HEAD, N_TRANSFORMER_BLOCK)
gpt2(x, torch.randint(0, 10, (4, 128), device=device))[1]

tensor(10.9969, device='cuda:0', grad_fn=<NllLossBackward0>)

# Training

In [139]:
LEARNING_RATE = 3e-4
EPOCH = 500
BATCH_SIZE = 32

In [140]:
def slice_train(data):
    x = data[:, :-1]
    y = data[:, 1:]

    return x, y

In [141]:
def get_batch(tensor, batch_size, blocksize):
    ids = torch.randint(0, len(tensor) - blocksize, size=(batch_size,), device=device)

    start_tok = torch.stack([tensor[id:id + blocksize + 1] for id in ids])

    return slice_train(start_tok)

In [142]:
tokenizer = tiktoken.get_encoding("gpt2")

path = Path("../dataset/input.txt")
text = path.read_text(encoding="utf-8")
size = len(text)

train_ids = text[:int(size * (1 - VAL_DATA_RATIO))]
val_ids = text[int(size * (1 - VAL_DATA_RATIO)):]

train_tok = tokenizer.encode(train_ids)
val_tok = tokenizer.encode(val_ids)

train_tok = torch.tensor(train_tok, dtype=torch.long, device=device)
val_tok = torch.tensor(val_tok, dtype=torch.long, device=device)

In [143]:
gpt2 = LLM(VOCAB_SIZE, BLOCK_SIZE, N_EMBD, N_HEAD, N_TRANSFORMER_BLOCK)
optimizer = torch.optim.AdamW(gpt2.parameters(), lr=LEARNING_RATE)

In [145]:
for i in range(EPOCH):
    gpt2.train()

    x, y = get_batch(train_tok, BATCH_SIZE, BLOCK_SIZE)
    optimizer.zero_grad()
    logits, loss = gpt2(x, y)

    loss.backward()
    optimizer.step()

    if (i % 50 == 0):
        nb_batch = 10
        sum_loss = 0

        with torch.no_grad():
            gpt2.eval()

            for _ in range (nb_batch):
                x_val, y_val = get_batch(val_tok, BATCH_SIZE, BLOCK_SIZE)
                _, loss_val = gpt2(x_val, y_val)
                sum_loss += loss_val.item()

        print(f"Epoch : {i} | train_loss : {loss.item():.5f} | val_loss : {sum_loss/nb_batch:.5f}")

Epoch : 0 | train_loss : 5.89314 | val_loss : 5.92103
Epoch : 50 | train_loss : 5.61450 | val_loss : 5.86754
Epoch : 100 | train_loss : 5.37605 | val_loss : 5.71663
Epoch : 150 | train_loss : 5.52217 | val_loss : 5.69303
Epoch : 200 | train_loss : 5.10616 | val_loss : 5.56577
Epoch : 250 | train_loss : 5.07080 | val_loss : 5.50040
Epoch : 300 | train_loss : 5.05571 | val_loss : 5.44200
Epoch : 350 | train_loss : 4.90451 | val_loss : 5.39484
Epoch : 400 | train_loss : 4.79586 | val_loss : 5.34670
Epoch : 450 | train_loss : 4.88869 | val_loss : 5.29437


# Inference

In [ ]:
TEMPURATURE = 1

In [ ]:
text = "ROMEO:\n"

ids = torch.tensor([tokenizer.encode(text)], dtype=torch.long, device=device)
res = gpt2.generate(ids, 100, temperature=TEMPURATURE)

for sequence in res:
    s = sequence.tolist()
    dec = tokenizer.decode(s)
    print(dec)

ROMEO:
Nor dead:
And only of thy news, and a half.

For we
I will if the white.

And if you'll do never that I say, for a
LADY ANNE:
The son.



POMPEY:


My lord, and one
GLOUCESTER:
Did at blood, they is peace, but
To be a one can he is sins.
And not?


